In [ ]:
import sys
import os
sys.path.append("../")
import torch
from PIL import Image
from configs import hf_token, HF_CACHE
os.environ['HF_HOME'] = HF_CACHE
from transformers import AutoProcessor, LlavaNextForConditionalGeneration, LlavaNextProcessor
%autosave 100

# LLAVA 

In [7]:
model_id = "llava-hf/llava-v1.6-vicuna-7b-hf"
model = LlavaNextForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    low_cpu_mem_usage=True,
    token=hf_token
).to("cuda")
processor = LlavaNextProcessor.from_pretrained(model_id)


Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.12s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [14]:
# --- Prepare image + question ---
image = Image.open("example.png")
question = "Describe the image."

conversation = [
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": question}
    ]}
]

chat_text = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False
)

inputs = processor(
    text=[chat_text],
    images=[image],
    return_tensors="pt"
).to(model.device)

# --- Containers for hook outputs ---
visual_features = {}

def hook_visual(module, input, output):
    visual_features["raw"] = output.last_hidden_state.detach().cpu()

# --- Register hooks ---
h1 = model.vision_tower.register_forward_hook(hook_visual)


In [15]:
generated_ids = model.generate(**inputs, max_new_tokens=50)

In [16]:
decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [17]:
decoded

'USER: \nDescribe the image. ASSISTANT: The image captures a lively scene in a park. The park is lush with green grass, and a brick path winds its way through the center, inviting visitors to explore. \n\nOn the left side of the image,'

In [19]:
visual_features["raw"].shape

torch.Size([5, 577, 1024])

# InternVL2-8B

In [2]:
import torch
from transformers import AutoModel, AutoTokenizer
from PIL import Image

# Load model
model_path = "OpenGVLab/InternVL2-8B"
model = AutoModel.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        use_flash_attn=True,
        trust_remote_code=True).eval().cuda()
processor = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=False)


/home/hice1/stekin6/.conda/envs/llamas/lib/python3.9/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


InternLM2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


In [5]:
model.mlp1

Sequential(
  (0): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
  (1): Linear(in_features=4096, out_features=4096, bias=True)
  (2): GELU(approximate='none')
  (3): Linear(in_features=4096, out_features=4096, bias=True)
)

In [7]:
from model_helper import load_image
from data_generator.data_helper import construct_open_ended_prompt
from configs import prompt_formats

# Load image
image = Image.open("example.png")
text = "What is happening in this image?"

res_dict = construct_open_ended_prompt(text, prompt_formats, processor=None)
generation_config = dict(max_new_tokens=100, return_dict_in_generate=False, output_scores=True)
pixel_values = load_image(image).to(torch.bfloat16).cuda()

output = model.chat(processor, pixel_values, res_dict["prompt"], generation_config)
# output_txt = processor.decode(output["sequences"][0], skip_special_tokens=True)

/home/hice1/stekin6/.conda/envs/llamas/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:818: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_scores` is. When `return_dict_in_generate` is not `True`, `output_scores` is ignored.
  warnings.warn(


In [7]:
output

'In the image, a group of people is gathered in a park-like setting. The scene appears to be a casual outdoor gathering or recreational event. Here are the details:\n\n1. **Bicycles**: Several bicycles are lying on the grass, suggesting that some people might have arrived by bike. The bicycles are parked haphazardly, indicating a relaxed and informal atmosphere.\n\n2. **People**: There are several individuals in the image. One person is standing near the bicycles'

In [6]:
# --- Containers for hook outputs ---
visual_features = {}
project_features = {}

def hook_visual(module, input, output):
    visual_features["raw"] = output.last_hidden_state.detach().cpu()

def hook_project(module, input, output):
    project_features["raw"] = output.detach().cpu()


# --- Register hooks ---
h1 = model.vision_model.register_forward_hook(hook_visual)
h2 = model.mlp1.register_forward_hook(hook_project)

In [9]:
visual_features["raw"].shape

torch.Size([13, 1025, 1024])

In [8]:
project_features["raw"].shape

torch.Size([13, 256, 4096])

In [10]:
model.get_input_embeddings()

Embedding(92553, 4096, padding_idx=2)

# DeepSeek VL2

In [1]:
import sys
import os
sys.path.append("../")
import torch
from PIL import Image
from configs import hf_token, HF_CACHE
os.environ['HF_HOME'] = HF_CACHE

from deepseek_vl2.models import DeepseekVLV2Processor, DeepseekVLV2ForCausalLM
from deepseek_vl2.utils.io import load_pil_images
from transformers import AutoModelForCausalLM


/home/hice1/stekin6/.conda/envs/deepseek/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_path = "deepseek-ai/deepseek-vl2-tiny"
processor = DeepseekVLV2Processor.from_pretrained(model_path)

vl_gpt: DeepseekVLV2ForCausalLM = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)
model = vl_gpt
model = model.to("cuda")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Add pad token = ['<｜▁pad▁｜>'] to the tokenizer
<｜▁pad▁｜>:2
Add image token = ['<image>'] to the tokenizer
<image>:128815
Add grounding-related tokens = ['<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>'] to the tokenizer with input_ids
<|ref|>:128816
<|/ref|>:128817
<|det|>:128818
<|/det|>:128819
<|grounding|>:128820
Add chat tokens = ['<|User|>', '<|Assistant|>'] to the tokenizer with input_ids
<|User|>:128821
<|Assistant|>:128822



In [20]:
image = Image.open("example.png").convert("RGB")
text = "What is happening in this image?"

In [ ]:
conversation = [
    {
        "role": "<|User|>",
        "content": f"<image>\n<|ref|>{text}<|/ref|>",
        "images": [image],
    },
    {"role": "<|Assistant|>", "content": ""},
]

prepare_inputs = processor(
    conversations=conversation,
    images=[image],
    force_batchify=True,
    system_prompt="").to(model.device)

inputs_embeds = model.prepare_inputs_embeds(**prepare_inputs)
output = model.language.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    pad_token_id=processor.tokenizer.eos_token_id,
    bos_token_id=processor.tokenizer.bos_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
    max_new_tokens=100,
    do_sample=True,
    # return_dict_in_generate=True,
    # output_scores=True,
    temperature=0.7
)
output_txt = processor.decode(output[0], skip_special_tokens=True)

AttributeError: 'BatchCollateOutput' object has no attribute 'items'

In [6]:
import torch
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

11.7
NVIDIA H100 80GB HBM3
(9, 0)


In [1]:
import torch
print("Torch version:", torch.__version__)
print("CUDA runtime version (compiled):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


Torch version: 2.5.1+cu121
CUDA runtime version (compiled): 12.1
CUDA available: True
GPU device: NVIDIA H200


In [15]:
model.device

device(type='cpu')

In [2]:
import torch

In [10]:
tensors = torch.load("../results/visual_features/okvqa/train/InternVL2-8B_vis_embed_3000.pt")
# tensors = torch.load("../results/visual_features/okvqa/train/llava-v1.6-vicuna-7b-hf_vis_embed.pt")

/tmp/ipykernel_4054581/2726546219.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensors = torch.load("../results/visual_features/okvqa/train/InternVL2-8B_vis_embed_300

In [11]:
len(tensors)

3003

In [12]:
tensors[2001].sum()

tensor(-132096., dtype=torch.bfloat16)

In [13]:
tensors[2002].sum()

tensor(-132096., dtype=torch.bfloat16)

In [16]:
a = tensors[:2000] + tensors[2002:]

In [21]:
torch.save(a, "../results/visual_features/okvqa/train/InternVL2-8B_vis_embed.pt")

In [1]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceM4/A-OKVQA")


/home/hice1/stekin6/.conda/envs/llamas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds["validation"]

Dataset({
    features: ['image', 'question_id', 'question', 'choices', 'correct_choice_idx', 'direct_answers', 'difficult_direct_answer', 'rationales'],
    num_rows: 1145
})

In [6]:
for i in ds["train"][:20]:
    print(i)

image
question_id
question
choices
correct_choice_idx
direct_answers
difficult_direct_answer
rationales
